# Studi Kasus: Kategori Mana yang Sebaiknya Diprioritaskan?

**Nama:** *tulis namamu*
**Tanggal:** 5 Desember 2026

> Sesi 12 · Studi kasus terpandu. Instruktur memimpin setiap langkah; kamu menjalankan sel yang sama dengan **fokus pilihanmu sendiri**. Simpan di portofolio sebagai `my_question/10_my_question.ipynb`.

## 1. Pertanyaan bisnis

Tim marketing TheLook punya anggaran promosi terbatas untuk kuartal depan dan hanya bisa memfokuskan kampanye pada **satu kategori produk**.

- **Pertanyaan:** Kategori produk mana yang sebaiknya diprioritaskan untuk kuartal depan?
- **Kenapa penting:** anggaran yang ditaruh di kategori yang tepat menghasilkan pendapatan dan keuntungan lebih besar.
- **Cara menjawab:** bandingkan setiap kategori dengan 4 ukuran:

| Ukuran | Pertanyaan yang dijawab |
| --- | --- |
| Porsi pendapatan | Seberapa besar kategori ini bagi bisnis **sekarang**? |
| Pertumbuhan | Apakah permintaannya **sedang naik**? |
| Margin % | Seberapa **untung** setiap penjualannya? |
| Tingkat retur | Seberapa sering barangnya **dikembalikan** (pendapatan yang hilang)? |

## 2. Pilih fokusmu

Setiap peserta memilih satu fokus. Itu sebabnya angka dan kesimpulanmu akan berbeda dari teman-temanmu.

- `FOKUS_DEPARTMENT`: `"Women"` atau `"Men"`.
- `FOKUS_NEGARA` *(opsional)*: satu negara, misalnya `"United States"`, `"China"`, `"Brasil"`. Biarkan `None` untuk semua negara.

> **VERSI INSTRUKTUR · KUNCI JAWABAN.** Jangan dibagikan ke peserta. Sel latihan dan PR sudah terisi; angka di jawaban teks ditandai `[...]` dan diisi dari output saat dijalankan.

In [ ]:
FOKUS_DEPARTMENT = "Women"   # <-- pilih "Women" atau "Men"
FOKUS_NEGARA = None          # <-- opsional, misalnya "China". None = semua negara

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from google.cloud import bigquery

PROJECT_ID = "isi-project-id-kamu"   # <-- GANTI dengan Project ID-mu
client = bigquery.Client(project=PROJECT_ID)
pd.set_option("display.float_format", "{:,.1f}".format)

def jalankan_query(sql):
    """Menjalankan query SQL di BigQuery dan mengembalikan hasilnya sebagai DataFrame."""
    return client.query(sql).to_dataframe()

WARNA_UTAMA = "#2a78d6"
WARNA_SOROT = "#eb6834"
WARNA_TEKS = "#5f6368"
plt.rcParams.update({
    "figure.figsize": (10, 6), "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e8eaed", "axes.axisbelow": True,
})
os.makedirs("images", exist_ok=True)

label_fokus = f"{FOKUS_DEPARTMENT}" + (f" di {FOKUS_NEGARA}" if FOKUS_NEGARA else "")
print("Fokus analisis:", label_fokus)

## 3. Mengambil data

Query ini sama dengan query final di `10_my_question.sql`. Fokusmu dimasukkan ke dalam SQL lewat f-string (`{...}`).

Satu baris hasil = satu kategori, dengan angka 12 bulan terakhir yang lengkap dan 12 bulan sebelumnya.

In [ ]:
filter_negara = f"AND u.country = '{FOKUS_NEGARA}'" if FOKUS_NEGARA else ""

SQL = f"""
WITH item AS (
  SELECT
    p.category,
    oi.status,
    oi.sale_price,
    p.cost,
    CASE
      WHEN DATE(oi.created_at) >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 12 MONTH)
        THEN 'terakhir'
      ELSE 'sebelumnya'
    END AS periode
  FROM `bigquery-public-data.thelook_ecommerce.order_items` AS oi
  JOIN `bigquery-public-data.thelook_ecommerce.products` AS p ON oi.product_id = p.id
  JOIN `bigquery-public-data.thelook_ecommerce.users` AS u ON oi.user_id = u.id
  WHERE p.department = '{FOKUS_DEPARTMENT}'
    {filter_negara}
    AND oi.status != 'Cancelled'
    AND DATE(oi.created_at) >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 24 MONTH)
    AND DATE(oi.created_at) < DATE_TRUNC(CURRENT_DATE(), MONTH)
)
SELECT
  category,
  SUM(CASE WHEN periode = 'terakhir' AND status != 'Returned' THEN sale_price ELSE 0 END) AS pendapatan,
  SUM(CASE WHEN periode = 'sebelumnya' AND status != 'Returned' THEN sale_price ELSE 0 END) AS pendapatan_sebelumnya,
  SUM(CASE WHEN periode = 'terakhir' AND status != 'Returned' THEN sale_price - cost ELSE 0 END) AS margin,
  COUNTIF(periode = 'terakhir') AS barang_terjual,
  COUNTIF(periode = 'terakhir' AND status = 'Returned') AS barang_retur,
  SUM(CASE WHEN periode = 'terakhir' AND status = 'Returned' THEN sale_price ELSE 0 END) AS pendapatan_hilang_retur
FROM item
GROUP BY category
ORDER BY pendapatan DESC
"""
kategori = jalankan_query(SQL).set_index("category")
kategori

## 4. Cek data

Sebelum menghitung, pastikan datanya masuk akal.

In [ ]:
print("Jumlah kategori        :", len(kategori))
print("Nilai kosong           :", kategori.isna().sum().sum())
print(f"Total pendapatan 12 bln: {kategori['pendapatan'].sum():,.0f} USD")
print(f"Total 12 bln sebelumnya: {kategori['pendapatan_sebelumnya'].sum():,.0f} USD")

## 5. Menghitung 4 ukuran

In [ ]:
kategori["porsi_pendapatan"] = kategori["pendapatan"] / kategori["pendapatan"].sum() * 100
kategori["pertumbuhan"] = (kategori["pendapatan"] / kategori["pendapatan_sebelumnya"] - 1) * 100
kategori["margin_persen"] = kategori["margin"] / kategori["pendapatan"] * 100
kategori["tingkat_retur"] = kategori["barang_retur"] / kategori["barang_terjual"] * 100

ukuran = ["porsi_pendapatan", "pertumbuhan", "margin_persen", "tingkat_retur"]
kategori[ukuran].sort_values("porsi_pendapatan", ascending=False)

### Kategori yang terlalu kecil

Kategori dengan porsi pendapatan di bawah 1% punya angka pertumbuhan yang "loncat-loncat": naik dari 100 USD ke 300 USD terlihat seperti tumbuh 200%, padahal nilainya kecil sekali. Kategori seperti ini kita pisahkan supaya tidak mengacaukan perbandingan.

In [ ]:
kecil = kategori[(kategori["porsi_pendapatan"] < 1) | (kategori["pendapatan_sebelumnya"] <= 0)]
utama = kategori.drop(kecil.index).copy()

print("Dipisahkan (terlalu kecil):", list(kecil.index))
print("Dianalisis                :", len(utama), "kategori")

## 6. Matriks prioritas

Kita bagi kategori menjadi 4 kelompok dengan dua pertanyaan: **porsinya besar?** dan **tumbuhnya cepat?** Batasnya adalah **median** semua kategori, sehingga setengah kategori ada di atas dan setengah di bawah.

| | Tumbuh cepat | Tumbuh lambat |
| --- | --- | --- |
| **Porsi besar** | **Andalan**: pertahankan dan perkuat | **Mesin Uang**: jaga, jangan boros anggaran |
| **Porsi kecil** | **Bintang Baru**: peluang terbaik untuk didorong | **Perlu Ditinjau**: evaluasi atau kurangi |

Ide ini mirip "matriks BCG", alat yang sering dipakai konsultan bisnis.

In [ ]:
batas_porsi = utama["porsi_pendapatan"].median()
batas_tumbuh = utama["pertumbuhan"].median()

def kuadran(baris):
    """Kelompok prioritas berdasarkan porsi pendapatan dan pertumbuhan."""
    besar = baris["porsi_pendapatan"] >= batas_porsi
    cepat = baris["pertumbuhan"] >= batas_tumbuh
    if besar and cepat:
        return "Andalan"
    elif besar:
        return "Mesin Uang"
    elif cepat:
        return "Bintang Baru"
    else:
        return "Perlu Ditinjau"

utama["kuadran"] = utama.apply(kuadran, axis=1)
print(f"Batas porsi: {batas_porsi:.1f}% | batas pertumbuhan: {batas_tumbuh:.1f}%")
utama.sort_values(["kuadran", "porsi_pendapatan"], ascending=[True, False])[ukuran + ["kuadran"]]

## 7. Memilih rekomendasi

Aturan yang kita pakai, ditulis dengan jelas supaya bisa diperiksa orang lain:

1. Ambil kategori **Bintang Baru** (kecil tapi tumbuh cepat). Kalau tidak ada, ambil **Andalan**.
2. Dari kandidat itu, pilih yang **margin %-nya tertinggi**, karena setiap dolar promosi menghasilkan keuntungan lebih besar.
3. Cek **tingkat returnya**. Kalau di atas median, catat sebagai risiko.

In [ ]:
kandidat = utama[utama["kuadran"] == "Bintang Baru"]
if kandidat.empty:
    kandidat = utama[utama["kuadran"] == "Andalan"]

rekomendasi = kandidat["margin_persen"].idxmax()
r = utama.loc[rekomendasi]
median_retur = utama["tingkat_retur"].median()

print("Kandidat   :", list(kandidat.index))
print("Rekomendasi:", rekomendasi)
r[ukuran]

## 8. Grafik 1: matriks prioritas

In [ ]:
fig, ax = plt.subplots()
lain = utama.drop(rekomendasi)
ax.scatter(lain["porsi_pendapatan"], lain["pertumbuhan"], s=70, color=WARNA_UTAMA, zorder=3)
ax.scatter(r["porsi_pendapatan"], r["pertumbuhan"], s=90, color=WARNA_SOROT, zorder=4)

# Label setiap titik. Kalau ada titik lain yang sangat dekat, labelnya digeser ke bawah
# supaya tidak bertumpuk.
lebar_x = utama["porsi_pendapatan"].max() - utama["porsi_pendapatan"].min()
lebar_y = utama["pertumbuhan"].max() - utama["pertumbuhan"].min()
sudah_dilabel = []
for nama, baris in utama.sort_values("pertumbuhan", ascending=False).iterrows():
    x, y = baris["porsi_pendapatan"], baris["pertumbuhan"]
    geser = sum(1 for px, py in sudah_dilabel
                if abs(x - px) < 0.08 * lebar_x and abs(y - py) < 0.06 * lebar_y)
    sudah_dilabel.append((x, y))
    ax.annotate(nama, (x, y),
                textcoords="offset points", xytext=(6, 4 - 12 * geser), fontsize=9,
                color=WARNA_SOROT if nama == rekomendasi else WARNA_TEKS,
                fontweight="bold" if nama == rekomendasi else "normal")

ax.axvline(batas_porsi, color="#bdc1c6", linestyle="--", linewidth=1)
ax.axhline(batas_tumbuh, color="#bdc1c6", linestyle="--", linewidth=1)

# Nama kuadran di setiap pojok
for x, y, teks, ha, va in [(0.99, 0.99, "Andalan", "right", "top"),
                           (0.01, 0.99, "Bintang Baru", "left", "top"),
                           (0.99, 0.01, "Mesin Uang", "right", "bottom"),
                           (0.01, 0.01, "Perlu Ditinjau", "left", "bottom")]:
    ax.text(x, y, teks, transform=ax.transAxes, ha=ha, va=va, color="#9aa0a6", fontsize=11)

ax.set_title(f"{rekomendasi}: porsi kecil, tumbuh {r['pertumbuhan']:.0f}%, margin {r['margin_persen']:.0f}%"
             if r["kuadran"] == "Bintang Baru" else
             f"{rekomendasi}: kategori Andalan dengan margin tertinggi ({r['margin_persen']:.0f}%)")
ax.set_xlabel("Porsi pendapatan 12 bulan terakhir (%)")
ax.set_ylabel("Pertumbuhan vs 12 bulan sebelumnya (%)")
fig.text(0.01, -0.02, f"Fokus: {label_fokus}. Garis putus-putus = median. Sumber: TheLook (BigQuery).",
         color=WARNA_TEKS, fontsize=9)
plt.tight_layout()
nama_file_1 = f"images/12_matriks_prioritas_{FOKUS_DEPARTMENT.lower()}.png"
plt.savefig(nama_file_1, dpi=150, bbox_inches="tight")
plt.show()

## 9. Grafik 2: margin % per kategori

In [ ]:
urut_margin = utama["margin_persen"].sort_values()

fig, ax = plt.subplots()
warna = [WARNA_SOROT if k == rekomendasi else WARNA_UTAMA for k in urut_margin.index]
ax.barh(urut_margin.index, urut_margin.values, color=warna, height=0.7)
for y, nilai in enumerate(urut_margin.values):
    ax.text(nilai + 0.3, y, f"{nilai:.1f}%", va="center", color=WARNA_TEKS, fontsize=9)

peringkat = list(urut_margin.index[::-1]).index(rekomendasi) + 1
ax.set_title(f"Margin {rekomendasi} {r['margin_persen']:.0f}%, peringkat {peringkat} dari {len(utama)} kategori")
ax.set_xlabel("Margin (%)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
nama_file_2 = f"images/12_margin_kategori_{FOKUS_DEPARTMENT.lower()}.png"
plt.savefig(nama_file_2, dpi=150, bbox_inches="tight")
plt.show()

## 10. Draf jawaban

Sel di bawah menyusun draf kesimpulan dari angka-angkamu. **Jangan disalin begitu saja ke README.** Baca, cocokkan dengan grafik, lalu tulis ulang dengan kata-katamu sendiri di bagian 11.

In [ ]:
total_hilang = utama["pendapatan_hilang_retur"].sum()
teks_risiko = (f"Tingkat returnya {r['tingkat_retur']:.1f}%, di atas median {median_retur:.1f}%, "
               f"jadi perlu dicek penyebab pengembaliannya (misalnya panduan ukuran)."
               if r["tingkat_retur"] > median_retur else
               f"Tingkat returnya {r['tingkat_retur']:.1f}%, tidak lebih tinggi dari median {median_retur:.1f}%.")

andalan = utama[utama["kuadran"] == "Andalan"].sort_values("porsi_pendapatan", ascending=False).index.tolist()

print(f"""JAWABAN
Untuk {label_fokus}, kategori yang paling layak diprioritaskan kuartal depan adalah {rekomendasi}.
Dalam 12 bulan terakhir kategori ini menyumbang {r['porsi_pendapatan']:.1f}% pendapatan, tumbuh {r['pertumbuhan']:.0f}%
(median semua kategori {batas_tumbuh:.0f}%), dan margin-nya {r['margin_persen']:.0f}%. {teks_risiko}

KONTEKS
Kategori Andalan (porsi besar dan tumbuh cepat): {', '.join(andalan) if andalan else '-'}.
Pendapatan yang hilang karena retur di kategori yang dianalisis: {total_hilang:,.0f} USD dalam 12 bulan.

REKOMENDASI
Fokuskan anggaran promosi kuartal depan pada {rekomendasi}, sambil menjaga kategori Andalan.
Ukur keberhasilannya dari pertumbuhan pendapatan {rekomendasi} kuartal depan dibanding kuartal yang sama tahun lalu.""")

## 11. Jawaban dengan kata-katamu sendiri

- **Jawaban:** *Contoh:* Untuk produk Women, saya merekomendasikan TheLook memprioritaskan **[kategori]** pada kuartal depan. Kategori ini baru menyumbang [X]% pendapatan, tetapi tumbuh [Y]% dalam 12 bulan terakhir, di atas median semua kategori ([Z]%), dan marginnya [M]%, termasuk yang tertinggi.
- **Apakah hasilnya sesuai dugaanmu?** *Contoh:* Saya menduga Jeans karena porsinya besar, tetapi Jeans termasuk Mesin Uang: besar namun pertumbuhannya di bawah median.
- **Rekomendasi:** *Contoh:* Fokuskan anggaran promosi kuartal depan pada [kategori], sambil menjaga kategori Andalan. Ukur keberhasilannya dari pertumbuhan pendapatan [kategori] dibanding kuartal yang sama tahun lalu.
- **Keterbatasan:** *Contoh:* Data TheLook sintetis; analisis belum memperhitungkan biaya promosi per kategori; pertumbuhan 12 bulan terakhir belum tentu berlanjut.

## 12. Unduh grafik

In [ ]:
from google.colab import files
files.download(nama_file_1)
files.download(nama_file_2)